In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import holoviews as hv
from holoviews.operation.datashader import rasterize
import hvplot.xarray # noqa
import panel as pn
import fsspec

pn.extension('tabulator')
hv.extension('bokeh')

In [2]:
DATA_URL = 'https://datasets.holoviz.org/miniscope/v1/real_miniscope_uint8.zarr/'
DATA_DIR = Path('/home/lag36/scratch/lag36/2025-01-29_Microscopy-Test-Case/')
DATA_FILENAME = Path(DATA_URL).name
DATA_PATH = DATA_DIR / DATA_FILENAME

print(f'Local Data Path: {DATA_PATH}')

DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download the data if it doesn't exist
if not DATA_PATH.exists():
    print(f'Downloading data to: {DATA_PATH}')
    ds_remote = xr.open_dataset(
        fsspec.get_mapper(DATA_URL), engine='zarr', chunks={}
    )
    ds_remote.to_zarr(str(DATA_PATH))  # Save locally
    print(f'Dataset downloaded to: {DATA_PATH}')
else:
    print(f'Data exists at: {DATA_PATH}')

Local Data Path: /home/lag36/scratch/lag36/2025-01-29_Microscopy-Test-Case/real_miniscope_uint8.zarr
Dataset downloaded to: /home/lag36/scratch/lag36/2025-01-29_Microscopy-Test-Case/real_miniscope_uint8.zarr


In [3]:
# Open the dataset from the local copy
ds = xr.open_dataset(
    DATA_PATH,
    engine='zarr',
    chunks={'frame': 400, 'height': -1, 'width': -1}  # Chunk by frames
)

# Access the variable 'varr_ref' which contains the image data
da = ds['varr_ref']
da

<xarray.DataArray 'varr_ref' (frame: 2000, height: 480, width: 752)> Size: 722MB
dask.array<open_dataset-varr_ref, shape=(2000, 480, 752), dtype=uint8, chunksize=(400, 480, 752), chunktype=numpy.ndarray>
Coordinates:
  * frame    (frame) int64 16kB 0 1 2 3 4 5 6 ... 1994 1995 1996 1997 1998 1999
  * height   (height) int64 4kB 0 1 2 3 4 5 6 7 ... 473 474 475 476 477 478 479
  * width    (width) int64 6kB 0 1 2 3 4 5 6 7 ... 745 746 747 748 749 750 751

In [27]:
hvplot_app = da.hvplot.image(
    groupby="frame",
    title='hvPlot App',
    cmap='viridis',
    clim=(0, 20),
    data_aspect=1,
    widget_location='bottom',
    rasterize=False
)

In [28]:
hvplot_app

Column
    [0] HoloViews(DynamicMap, sizing_mode='fixed', widget_location='bottom')
    [1] WidgetBox(align=('center', 'end'))
        [0] DiscreteSlider(formatter='%d', name='frame', options={'0': np.int64(0), ...}, value=np.int64(0))

In [5]:
dd.DataFrame

dask.dataframe.dask_expr._collection.DataFrame

In [24]:
da_subset = da.isel(frame=slice(20, 40))

da_subset.hvplot.image(
    dynamic=False,  # Embeds all frames in webpage. Only do this with a few frames.
    groupby="frame",
    title='Use my widget on a static website!',
    cmap='viridis',
    clim=(0, 20),
    data_aspect=1,
    widget_location='bottom',
)

Column
    [0] HoloViews(HoloMap, sizing_mode='fixed', widget_location='bottom')
    [1] WidgetBox(align=('center', 'end'))
        [0] DiscreteSlider(formatter='%d', name='frame', options={'20': np.int64(20), ...}, value=np.int64(20))

In [11]:
max_proj = da.max('frame').compute().astype(np.float32)
img_max_proj = max_proj.hvplot.image(
    title='Max Over Time',
    cmap="magma",
    clim=(0,20),
    data_aspect=1,
)
img_max_proj

:Image   [width,height]   (varr_ref)

In [13]:
pn.Row(hvplot_app, img_max_proj)

Row
    [0] Column
        [0] HoloViews(DynamicMap, sizing_mode='fixed', widget_location='bottom')
        [1] WidgetBox(align=('center', 'end'))
            [0] DiscreteSlider(formatter='%d', name='frame', options={'0': np.int64(0), ...}, value=np.int64(0))
    [1] HoloViews(Image, sizing_mode='fixed')

In [14]:
def plot_image(frame):
    return hv.Image(da.sel(frame=frame), kdims=["width", "height"]).opts(
        title=f'Frame = {frame}',
        frame_height=da.sizes['height'],
        frame_width=da.sizes['width'],
        cmap='Viridis',
        clim=(0, 20),
        colorbar=True,
        tools=['hover', 'crosshair'],
        toolbar='right',
        apply_hard_bounds=True,
        scalebar=True,
        scalebar_unit=("µm", "m"),  # Each pixel is about 1 µm in this dataset
        scalebar_opts={
            'background_fill_alpha': 0.5,
            'border_line_color': None,
            'bar_length': 0.10,
        }
    )

frame_player = pn.widgets.Player(
    length=len(da.coords['frame']),
    interval=100,  # Inter-frame-interval in milliseconds
    value=20,     # Arbitrary starting frame
    show_loop_controls=False,
    align='center',
    scale_buttons=0.9,
    sizing_mode='stretch_width',
    show_value=True,
    value_align='center',
    visible_buttons=['slower', 'previous', 'pause', 'play', 'next', 'faster'],
)

In [15]:
main_view = hv.DynamicMap(pn.bind(plot_image, frame_player.param.value_throttled))

In [19]:
# Compute the maximum projection over time
max_proj_time = da.max('frame').compute().astype(np.float32)

# Create the maximum projection image
img_max_proj_time = hv.Image(
    max_proj_time, ['width', 'height'], label='Max Over Time'
).opts(
    cmap='magma',
)

# Opacity slider for the overlay
opacity_slider = pn.widgets.FloatSlider(
    start=0, end=1, step=0.1, value=0.3, name='Opacity', align='center', sizing_mode='stretch_width'
)

# Link the slider value to the overlay's alpha (opacity)
opacity_slider.jslink(img_max_proj_time, value='glyph.global_alpha')

Link(args={}, bidirectional=False, code=None, name='Link02234', properties={'value': 'glyph.global_alpha'})

In [20]:
main_view

:DynamicMap   []
   :Image   [width,height]   (varr_ref)